# Agri Vision: combined EfficientDet continuation - re19gmail

Use hannanre19@gmail.com (different from the YOLO account). Keep the old 4gmail and 71gmail EfficientDet jobs stopped. Select T4 GPU and Run all. This notebook contains the verified combination of trials 1-11 from 71gmail and trials 12-16 from 4gmail.

15 screening trials are complete. Trial 16 has 3 committed rounds in this snapshot. Completed work is skipped; an unfinished round repeats from its last committed global checkpoint. Once all 16 five-round trials are complete, the global top two continue to round 10. The selected configuration then starts its fresh 30-round final run automatically. No manual model switching is required.

The original dataset, code, common initialization, hardware checks and optimized FP32 execution profile are retained. This preparation does not start GPU training.


In [ ]:
from pathlib import Path
DRIVE_ROOT = Path("/content/drive/MyDrive/Agri Vision")
WORKSPACE = Path("/content/agrivision_grid_v1")
OUTPUT = DRIVE_ROOT / "grid_search" / "grid_v1_20261017"
MODELS = ['efficientdet_d0']
WORKERS = 2
EXPECTED_DATASET_SHA256 = "7a2920e4a4e0405ee157e9b40ac6306b54d6933b20a306b44a4b7c6871f20868"
EXPECTED_CODE_SHA256 = "5453a56a97c2aca78224cec346c259e5f8a0a5eda2a93eb17b3677f06689c049"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import hashlib, json, shutil, zipfile
assert DRIVE_ROOT.is_dir(), "Open this notebook as hannanre19@gmail.com; the copied Agri Vision folder must be in My Drive"
assert (DRIVE_ROOT / "combined_effdet_re19gmail.json").is_file(), "Wrong Agri Vision folder: account migration marker missing"

def sha(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

WORKSPACE.mkdir(parents=True, exist_ok=True)
dataset_zip = WORKSPACE.parent / "non_chili_v1_colab_dataset.zip"
source_zip = DRIVE_ROOT / dataset_zip.name
if source_zip.is_file():
    if not dataset_zip.is_file() or sha(dataset_zip) != EXPECTED_DATASET_SHA256:
        shutil.copyfile(source_zip, dataset_zip)
else:
    if not dataset_zip.is_file() or sha(dataset_zip) != EXPECTED_DATASET_SHA256:
        parts_dir = DRIVE_ROOT / "non_chili_v1_upload_parts"
        manifest = json.loads((parts_dir / "parts.json").read_text())
        assert manifest["joined_sha256"] == EXPECTED_DATASET_SHA256, "Wrong dataset parts manifest"
        specs = manifest["parts"]
        assert specs and [p["name"] for p in specs] == [dataset_zip.name + f".part{i:02d}" for i in range(1, len(specs)+1)]
        partial = dataset_zip.with_suffix(".partial")
        with partial.open("wb") as output:
            for index, spec in enumerate(specs, 1):
                path = parts_dir / spec["name"]
                assert path.is_file() and path.stat().st_size == spec["bytes"], f"Missing/damaged part {index}"
                with path.open("rb") as stream:
                    shutil.copyfileobj(stream, output, 16*1024*1024)
                print(f"Dataset part {index}/{len(specs)} copied", flush=True)
        assert sha(partial) == EXPECTED_DATASET_SHA256, "Dataset parts damaged"
        partial.replace(dataset_zip)
assert sha(dataset_zip) == EXPECTED_DATASET_SHA256
dataset_marker = WORKSPACE / ".dataset_verified"
if not dataset_marker.exists() or dataset_marker.read_text() != EXPECTED_DATASET_SHA256:
    with zipfile.ZipFile(dataset_zip) as bundle:
        for name in bundle.namelist():
            assert (WORKSPACE/name).resolve().is_relative_to(WORKSPACE.resolve())
        bundle.extractall(WORKSPACE)
    dataset_marker.write_text(EXPECTED_DATASET_SHA256)
code_zip = WORKSPACE.parent / "agrivision_grid_search_v1_code.zip"
shutil.copyfile(DRIVE_ROOT/code_zip.name, code_zip)
assert sha(code_zip) == EXPECTED_CODE_SHA256, "Wrong grid-search code bundle"
with zipfile.ZipFile(code_zip) as bundle:
    for name in bundle.namelist():
        assert (WORKSPACE/name).resolve().is_relative_to(WORKSPACE.resolve())
    bundle.extractall(WORKSPACE)
print("Verified local dataset and isolated grid-search code.")


In [ ]:
import os, subprocess, sys
os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE))
import torch, torchvision
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > T4 GPU"
assert "T4" in torch.cuda.get_device_name(0), "Use T4 for the comparison"
torch_before, vision_before = torch.__version__, torchvision.__version__
print("GPU:", torch.cuda.get_device_name(0), "torch:", torch_before, "torchvision:", vision_before)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-r", "requirements-grid.txt"], check=True)
# Preserve the existing CUDA packages while installing supporting libraries.
subprocess.run([sys.executable, "-m", "pip", "install", "huggingface_hub", "safetensors", "packaging", "tqdm", "omegaconf==2.4.0",
                "matplotlib", "opencv-python", "requests", "scipy", "psutil", "polars>=0.20", "ultralytics-thop>=2.0.18"], check=True)
assert torch.__version__ == torch_before and torchvision.__version__ == vision_before
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_grid_search.py", "-v"], check=True)


In [ ]:
# Validate the combined snapshot and exact resume environment before training.
marker_path = DRIVE_ROOT / "combined_effdet_re19gmail.json"
assert sha(marker_path) == "ee30b3cc4ca7f32e5dd3682e9e4c68097b07196b632ec17b1d975ab0df557545", "Wrong combined-search marker"
snapshot = json.loads(marker_path.read_text())
model_folder = OUTPUT / "efficientdet_d0"
initial = model_folder / "initial.pt"
assert sha(initial) == snapshot["initial_checkpoint_sha256"], "Common initialization differs"
assert sha(OUTPUT/"plan.json") == snapshot["plan_sha256"], "Saved search plan differs"
assert sha(model_folder/"execution_profile.json") == snapshot["execution_profile_sha256"], "Execution profile differs"
from importlib.metadata import version
reference = json.loads((model_folder/"trials/trial_01/settings.json").read_text())
actual = {"gpu": torch.cuda.get_device_name(0), "torch": str(torch.__version__),
    "packages": {name: version(name) for name in reference["hardware"]["packages"]}}
assert actual == reference["hardware"], f"Resume environment differs: {actual}"
assert set(snapshot["trials"]) == {f"trial_{n:02d}" for n in range(1,17)}
for name, source in sorted(snapshot["trials"].items()):
    trial = model_folder/"trials"/name
    settings = json.loads((trial/"settings.json").read_text())
    assert sha(trial/"settings.json") == source["settings_sha256"], f"Settings changed: {name}"
    assert sha(model_folder/"configs"/f"{name}.json") == source["config_sha256"], f"Configuration changed: {name}"
    assert settings["initial_checkpoint"] == str(initial)
    assert settings["initial_checkpoint_sha256"] == snapshot["initial_checkpoint_sha256"]
    assert settings["hardware"] == actual
    latest = json.loads((trial/"latest.json").read_text())
    assert latest["round"] >= source["round"], f"Committed progress lost: {name}"
    checkpoint = (trial/latest["checkpoint"]).resolve()
    assert checkpoint.is_relative_to(trial.resolve())
    assert sha(checkpoint) == latest["checkpoint_sha256"], f"Damaged checkpoint: {name}"
    if latest["round"] == source["round"]:
        assert latest["checkpoint_sha256"] == source["checkpoint_sha256"]
    for relative, expected in source["metric_sha256"].items():
        assert sha(trial/relative) == expected, f"Historical metric changed: {name}/{relative}"
    assert all((trial/"results/metrics"/f"round_{n:03d}.json").is_file() for n in range(1,latest["round"]+1))
    print(name, "verified saved round", latest["round"], flush=True)
print("All 16 combined trials verified. Remaining screening, global confirmation and final training will resume.", flush=True)


## Automatic search, confirmation and final training
Each model's `preflight.json` records batch/resolution compatibility. `screening_results.csv` ranks all sixteen five-round trials, and `finalist_results.csv` ranks the two ten-round finalists. `selected_configuration.json` identifies the winner. The runner then trains that configuration from scratch through round 30 under `final_training/`, retaining validation-best and latest weights.


In [ ]:
extension = DRIVE_ROOT / "efficientdet_fp32_continuation_v1.zip"
assert sha(extension) == "35e010acf0d7bfebd3ff585d7cc33bdabd9bc009371d9e01b2fcc4f27ca5820e", "Wrong optimized continuation bundle"
with zipfile.ZipFile(extension) as archive:
    for name in archive.namelist():
        assert (WORKSPACE / name).resolve().is_relative_to(WORKSPACE.resolve())
        assert name.startswith(("continuation/", "benchmarks/")), "Extension must not replace original training modules"
    archive.extractall(WORKSPACE)
print("Verified optimized FP32 extension; continuing existing EfficientDet search.", flush=True)
command = [sys.executable, "-u", "-m", "continuation.search", "--root", str(WORKSPACE),
           "--output", str(OUTPUT), "--models", *MODELS, "--workers", str(WORKERS)]
env = dict(os.environ, PYTHONUNBUFFERED="1")
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1, env=env) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    returncode = process.wait()
if returncode:
    raise subprocess.CalledProcessError(returncode, command)
print("Completed models:", MODELS)
print("Results:", OUTPUT)


## Resume and results
Always reopen this notebook as hannanre19@gmail.com, select T4 and Run all. All checkpoints/settings and paths remain under My Drive/Agri Vision/grid_search/grid_v1_20261017/efficientdet_d0/. Keep this folder named Agri Vision.

The copied snapshot marker is fixed. Its checks accept later committed rounds while preserving original source settings and metrics. Completed screening trials are skipped after identity checks. screening_results.csv ranks all completed screening trials, finalist_results.csv ranks the global finalists, and final_training/ stores the selected fresh 30-round run with best/latest weights. The notebook exits automatically after the final 30 rounds.

Do not run old EfficientDet notebooks concurrently. The archived partial trial-12 copy on 71gmail is deliberately excluded; the completed 4gmail trial 12 is the official copy. This experiment uses custom sequential three-client FedAvg, not Flower.
